# Веб-скрейпинг сайта «Терновое королевство»

Ноутбук для Google Colab. Сайт — одностраничный тренажёр по математике для 2 класса (`ternovoe-korolevstvo.html`).

Ноутбук идёт по шагам главы 10 гайда. В главе код написан на R, здесь те же шаги сделаны на Python:

| Шаг в гайде | В гайде (R) | Здесь (Python) |
|---|---|---|
| Запрос с User-Agent | `polite::bow()`, `scrape()` | `requests.get(..., headers=...)` |
| Выбор элементов | `html_elements()` | `soup.select()` — CSS-селекторы, `tree.xpath()` — XPath |
| Текст и атрибуты | `html_text()`, `html_attr()` | `.get_text()`, `["атрибут"]` |
| Таблица | `data.frame()`, `rbind()` | `pandas.DataFrame`, `pd.concat()` |
| Сохранение | `write_csv()`, `write_xlsx()` | `to_csv()`, `to_excel()` |
| Динамические страницы | RSelenium | Selenium |

**План**

1. Загрузить страницу запросом.
2. Проверить, статическая она или динамическая.
3. Открыть страницу в браузере через Selenium.
4. Собрать полянки-игры CSS-селекторами и XPath.
5. Нажать кнопку «Теория» и собрать правила.
6. Пройти циклом по вкладкам «Книги заклинаний» и собрать таблицу умножения.
7. Собрать задания, которые игра создаёт случайно.
8. Достать из кода страницы то, чего нет на экране.
9. Проверить и сохранить результат.

**Этика:** страница своя, запрос один, в запросе указан `User-Agent`, стоит таймаут. Для чужих сайтов перед сбором проверяют `robots.txt` и условия использования, а между запросами делают паузы.

## 1. Загрузка страницы запросом

In [ ]:
# Ссылка на HTML-страницу сайта в этом же репозитории
URL = "https://raw.githubusercontent.com/psdvoretskaya/digital-intelligence/main/ternovoe-korolevstvo%20(1).html"

# Если URL сделать пустым, ноутбук возьмёт файл с таким именем или попросит загрузить его с компьютера.
LOCAL_FILE = "ternovoe-korolevstvo.html"


In [ ]:
import os, re, json, time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from lxml import html as lxml_html

HEADERS = {"User-Agent": "Mozilla/5.0 (study project; Google Colab)"}

def load_html(url, local_file):
    """Возвращает (текст страницы, путь к локальной копии)."""
    if url:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()              # остановиться, если сервер ответил ошибкой
        r.encoding = "utf-8"
        print("Код ответа:", r.status_code, "| размер:", len(r.text), "символов")
        with open(local_file, "w", encoding="utf-8") as f:
            f.write(r.text)               # копия нужна, чтобы открыть страницу в браузере
        return r.text, local_file
    if not os.path.exists(local_file):
        from google.colab import files    # работает только в Colab
        uploaded = files.upload()
        local_file = next(iter(uploaded))
    with open(local_file, encoding="utf-8") as f:
        text = f.read()
    print("Файл:", local_file, "| размер:", len(text), "символов")
    return text, local_file

source, HTML_PATH = load_html(URL, LOCAL_FILE)

## 2. Статическая страница или динамическая?

In [ ]:
soup = BeautifulSoup(source, "html.parser")

page_info = {
    "Заголовок: soup.title": soup.title.get_text(strip=True),
    "Язык: html[lang]": soup.html["lang"],
    "Шрифты: link[href]": "; ".join(re.findall(r"family=([^&:]+)", " ".join(l.get("href", "") for l in soup.select("link[href]")))),
    "Тегов всего": len(soup.find_all(True)),
    "Кнопок: button": len(soup.select("button")),
    "Заголовков: h1, h2, h3": len(soup.select("h1, h2, h3")),
    "Текст в контейнере div#app": repr(soup.select_one("div#app").get_text(strip=True)),
}
pd.DataFrame(page_info.items(), columns=["Что проверяем", "Результат"])

В исходном коде нет ни одной кнопки и ни одного заголовка, а контейнер `div#app` пустой. Текст сайта появляется только после выполнения JavaScript. Значит, страница **динамическая** (раздел 05 гайда): селекторы по ответу на запрос ничего не найдут, нужен браузер.

## 3. Selenium: сессия браузера

In [ ]:
# Установка Selenium и браузера Chrome в Colab (1–2 минуты)
!pip -q install selenium
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get -qq install -y ./google-chrome-stable_current_amd64.deb > /dev/null

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions()
options.add_argument("--headless=new")            # браузер без окна: в Colab экрана нет
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1200,900")

driver = webdriver.Chrome(options=options)        # создаём сессию
wait = WebDriverWait(driver, 15)

driver.get("file://" + os.path.abspath(HTML_PATH))   # переход по ссылке (в гайде — navigate)

# Сайт спрашивает имя игрока: вводим текст в поле и нажимаем Enter (в гайде — sendKeysToElement)
name_field = wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, "input#hName")))
name_field.send_keys("Скрейпер", Keys.ENTER)

wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.node")))
print("Открыта страница:", driver.title)

## 4. Полянки: CSS-селекторы и XPath

Берём код страницы после работы JavaScript (`driver.page_source`, в гайде — `getPageSource`) и разбираем его теми же инструментами, что и статическую страницу.

Карточка полянки в коде выглядит так:

```html
<button class="node" data-k="0" title="Что такое умножение: по a взять b раз">
  <span class="num">1</span><span class="stk">👝</span><b>Волшебные мешочки</b>
  <span class="stars" aria-label="0 из 3 звёзд">…</span>
</button>
```

In [ ]:
home = BeautifulSoup(driver.page_source, "html.parser")
print("Кнопок теперь:", len(home.select("button")), "| заголовок h1:", home.select_one("h1").get_text())

rows = []
for card in home.select("section#board button.node"):          # тег + класс внутри контейнера с id
    rows.append({
        "№": int(card.select_one("span.num").get_text()),
        "Значок": card.select_one("span.stk").get_text(),
        "Название": card.select_one("b").get_text(),
        "Описание": card["title"],                             # значение атрибута
    })
stations = pd.DataFrame(rows)

SPEED = {"Лабиринт дракончика", "Перехват в облаках", "Ударь крота", "Парочки-перевёртыши", "Сортировка"}
stations["Тип"] = stations["Название"].map(lambda n: "аркада" if n in SPEED else "задания")
print("Полянок:", len(stations))
stations

In [ ]:
# То же самое через XPath
tree = lxml_html.fromstring(driver.page_source)

names_xpath = tree.xpath("//button[contains(@class,'node')]/b/text()")          # относительный путь + условие
about_xpath = tree.xpath("//button[contains(@class,'node')]/@title")            # атрибут
parent_id   = tree.xpath("//button[@data-k='0']/parent::*/@id")                 # ось parent: в каком контейнере лежит карточка
menu        = tree.xpath("//button[starts-with(@id,'th') or starts-with(@id,'book') or starts-with(@id,'log')]/text()")

print("Совпадает с CSS-селекторами:", names_xpath == list(stations["Название"]) and about_xpath == list(stations["Описание"]))
print("Родительский контейнер карточки:", parent_id)
print("Кнопки меню:", menu)

## 5. Клик по кнопке и сбор теории

In [ ]:
driver.find_element(By.CSS_SELECTOR, "button#thBtn").click()                    # в гайде — clickElement
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "section.card.th")))

page = BeautifulSoup(driver.page_source, "html.parser")
rows = []
for card in page.select("section.card.th"):
    title = card.select_one("h3").get_text()
    rules = [r.get_text(" ", strip=True) for r in card.select("div.rule")]      # формулы в рамке
    examples = [e.get_text(" ", strip=True) for e in card.select("div.ex p")]   # p внутри div.ex
    text = " ".join(card.get_text(" ", strip=True).split())[len(title):].strip()
    rows.append({"Тема": title, "Формулы": " | ".join(rules), "Примеры": " | ".join(examples),
                 "Текст": text, "Слов": len(re.findall(r"[А-Яа-яЁё]+", text))})
theory = pd.DataFrame(rows)

driver.find_element(By.CSS_SELECTOR, "button#back").click()                     # назад на карту
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.node")))
print("Тем теории:", len(theory))
theory

## 6. Цикл по вкладкам: таблица умножения

В гайде цикл `for` перебирает страницы новостей, меняя номер в ссылке. У этого сайта ссылка одна, а «страницы» — это вкладки «на 2 … на 9» в «Книге заклинаний». Цикл нажимает вкладки по очереди, с каждой собирает данные и добавляет их в общую таблицу (в гайде — `rbind`, здесь — `pd.concat`).

In [ ]:
driver.find_element(By.CSS_SELECTOR, "button#bookBtn").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.tab")))

multiplication = pd.DataFrame()      # пустая таблица до цикла
tips = []

for t in range(2, 10):
    driver.find_element(By.CSS_SELECTOR, f"button.tab[data-t='{t}']").click()
    wait.until(EC.text_to_be_present_in_element((By.CSS_SELECTOR, "section.card h3"), f"на {t}"))
    time.sleep(0.2)                  # пауза между действиями
    page = BeautifulSoup(driver.page_source, "html.parser")

    expressions = [b.get_text() for b in page.select("div#mtiles button.mtile")]              # «3 · 7»
    answers = [int(s.get_text().replace("=", "")) for s in page.select("div.mslots .mslot")]  # «= 21»
    # выражения и ответы на странице перемешаны; после сортировки они встают парами
    expressions.sort(key=lambda e: int(e.split("·")[1]))
    answers.sort()

    part = pd.DataFrame({"Таблица на": t, "Выражение": expressions, "Ответ": answers})
    multiplication = pd.concat([multiplication, part], ignore_index=True)
    tips.append({"Таблица на": t, "Приём": page.select_one("p.tip").get_text(" ", strip=True).replace("Секрет: ", "")})

tips = pd.DataFrame(tips)
driver.find_element(By.CSS_SELECTOR, "button#back").click()
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.node")))
print("Строк в таблице умножения:", len(multiplication), "| приёмов:", len(tips))
multiplication.head(12)

In [ ]:
tips

## 7. Задания, которые игра создаёт случайно

Условия задач сайт не хранит: он собирает их генераторами при каждом запуске. Через Selenium можно выполнить JavaScript прямо на странице и получить результат. Цикл проходит по 11 полянкам и берёт с каждой по `SAMPLES` заданий.

In [ ]:
GENERATORS = {"Волшебные мешочки": "genBags", "Грядки-прямоугольники": "genRect", "Зеркало перестановки": "genMirror",
              "Школа приёмов": "genTricks", "Заклинание скобок": "genParen", "Составь выражение": "genBuild",
              "Магический квадрат": "genMagic", "Загадки Цветочного города": "genLogic", "Круги Эйлера": "genEuler",
              "Кухня Фрекен Бок": "genKitchen", "Турнир Тучки-Ворчуньи": "genBoss"}
SAMPLES = 20

JS = """
const fn = arguments[0], n = arguments[1], out = [];
for (let i = 0; i < n; i++) {
  const q = window.__tern[fn](i);
  out.push({kind: q.kind, prompt: q.prompt, plain: q.plain || '',
            answer: q.answer == null ? '' : String(q.answer), solution: q.explain || q.tx || ''});
}
return out;
"""

def text_of(fragment):
    """HTML-фрагмент -> обычный текст."""
    return " ".join(BeautifulSoup(str(fragment), "html.parser").get_text(" ").split())

KIND = {"choice": "выбор из вариантов", "input": "ввод числа", "custom": "задание в несколько шагов"}
rows = []
for station, fn in GENERATORS.items():
    for q in driver.execute_script(JS, fn, SAMPLES):
        rows.append({"Полянка": station, "Вид ответа": KIND[q["kind"]],
                     "Задание": q["plain"] or text_of(q["prompt"]),
                     "Ответ": text_of(q["answer"]), "Решение": text_of(q["solution"])})
generated = pd.DataFrame(rows)

driver.quit()                        # закрываем браузер: дальше он не нужен
print("Собрано заданий:", len(generated), "| разных:", generated["Задание"].nunique())
generated.groupby("Полянка").head(2)

## 8. Данные, которых нет на экране

Часть данных сайт держит в коде и показывает только во время игры: олимпиадные и логические задачи с ответами, список значков, палитру. Ни CSS-селектор, ни XPath до них не дотянутся, потому что это не теги, а текст внутри `<script>` и `<style>`. Здесь помогают регулярные выражения по исходному коду из раздела 1.

In [ ]:
script = max((s.get_text() for s in soup.select("script")), key=len)    # самый длинный скрипт — код игры
style = soup.select_one("style").get_text()

def block(name):
    """Текст объявления `const NAME=...;` из кода страницы."""
    start = script.index(f"const {name}=")
    return script[start:script.index(";\n", start)]

olympiad = pd.DataFrame(
    [{"Задача": q, "Ответ": int(a), "Решение": x, "Слов в условии": len(q.split())}
     for q, a, x in re.findall(r"\{q:'([^']+)',a:(\d+),x:'([^']+)'\}", block("OLY"))])

lst = lambda s: re.findall(r"'([^']*)'", s)
rows = []
for t, txt, cl, r, c, sol in re.findall(
        r"\{t:'([^']+)',txt:'([^']+)',cl:\[([^\]]+)\],rows:\[([^\]]+)\],cols:\[([^\]]+)\],sol:\[([^\]]+)\]\}", block("PUZ")):
    who, where, sol = lst(r), lst(c), [int(x) for x in sol.split(",")]
    rows.append({"Название": t, "Условие": txt, "Подсказки": " ".join(lst(cl)), "Число подсказок": len(lst(cl)),
                 "Размер таблицы": f"{len(who)}×{len(where)}",
                 "Ответ": "; ".join(f"{w} — {where[s]}" for w, s in zip(who, sol))})
logic = pd.DataFrame(rows)

badges = pd.DataFrame(re.findall(r"\['(\w+)','([^']+)','([^']+)',pf=>", block("BADGES")), columns=["id", "Значок", "Название"])

root = re.search(r":root\{(.*?)\}", style, re.S).group(1)
colors = pd.DataFrame(re.findall(r"--([\w-]+):\s*(#[0-9a-fA-F]{3,8})", root), columns=["Переменная CSS", "Цвет"])

print("Олимпиадных задач:", len(olympiad), "| логических:", len(logic), "| значков:", len(badges), "| цветов:", len(colors))
olympiad.head()

In [ ]:
logic

## 9. Проверка, короткий анализ и сохранение

In [ ]:
tables = {"Полянки": stations, "Теория": theory, "Таблица умножения": multiplication, "Приёмы": tips,
          "Задания": generated, "Олимпиада": olympiad, "Логика": logic, "Значки": badges, "Цвета": colors}

check = pd.DataFrame([{"Таблица": name, "Строк": len(df), "Столбцов": df.shape[1],
                       "Пустых ячеек": int((df.isna() | (df == "")).sum().sum())} for name, df in tables.items()])
print("Таблица умножения собрана верно:",
      all(eval(e.replace("·", "*")) == a for e, a in zip(multiplication["Выражение"], multiplication["Ответ"])))
check

In [ ]:
print("Полянки по типам:")
display(stations["Тип"].value_counts().to_frame("Полянок"))

print("Задания по виду ответа:")
display(pd.crosstab(generated["Полянка"], generated["Вид ответа"]))

print("Темы теории по объёму текста:")
display(theory[["Тема", "Слов"]].sort_values("Слов", ascending=False))

In [ ]:
OUT = "ternovoe_data"
os.makedirs(OUT, exist_ok=True)

with pd.ExcelWriter(f"{OUT}/ternovoe.xlsx") as xl:
    for name, df in tables.items():
        df.to_csv(f"{OUT}/{name}.csv", index=False, encoding="utf-8-sig")   # utf-8-sig, чтобы Excel открыл кириллицу
        df.to_excel(xl, sheet_name=name, index=False)

with open(f"{OUT}/ternovoe.json", "w", encoding="utf-8") as f:
    json.dump({name: json.loads(df.to_json(orient="records", force_ascii=False)) for name, df in tables.items()},
              f, ensure_ascii=False, indent=2)

print(sorted(os.listdir(OUT)))

In [ ]:
# Скачать всё одним архивом (только в Colab)
import shutil
shutil.make_archive(OUT, "zip", OUT)
try:
    from google.colab import files
    files.download(f"{OUT}.zip")
except ImportError:
    print("Архив лежит рядом с ноутбуком:", f"{OUT}.zip")